# Chapter 5 - Processed/Distinct Ratio: rapporto campioni processati / distinti processati

Richiesta relatore: tracciare il rapporto
\(ho_t = t / F_0(t)\), con:
- \(t\) = `number_of_elements_processed`
- \(F_0(t)\) = `f0_mean_t`

Versione solo lineare.
Output in `thesis/figures/results`:
- `ratio_processed_vs_distinct_streaming_linear.png`
- `ratio_processed_vs_distinct_streaming_boundary_linear.png`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Repository root non trovata: esegui il notebook dalla repo o da una sua sottocartella')


REPO = find_repo_root(Path.cwd().resolve())
RESULTS_ROOT = REPO / 'results' / 'prefix_constant_rho' / 'streaming'
OUT_DIR = REPO / 'thesis' / 'figures' / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_N = 10_000_000
TARGET_SEED = 21_041_998

TARGET_PARAMS = {
    'HyperLogLog++': 'k=14',
    'HyperLogLog': 'k=14,L=32',
    'LogLog': 'k=14,L=32',
    'Probabilistic Counting': 'L=31',
}

print('REPO:', REPO)
print('RESULTS_ROOT:', RESULTS_ROOT)
print('OUT_DIR:', OUT_DIR)


In [ ]:
paths = sorted(RESULTS_ROOT.glob('**/results_streaming.csv'))
if not paths:
    raise FileNotFoundError(f'Nessun CSV trovato in: {RESULTS_ROOT}')

frames = []
for p in paths:
    df = pd.read_csv(p)
    df['source_path'] = str(p)
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True)

sel = all_df[(all_df['mode'] == 'streaming') &
             (all_df['sample_size'] == TARGET_N) &
             (all_df['seed'] == TARGET_SEED)].copy()

for algo, params in TARGET_PARAMS.items():
    sel = sel[~((sel['algorithm'] == algo) & (sel['params'] != params))]

sel = sel[sel['algorithm'].isin(list(TARGET_PARAMS.keys()))].copy()
sel = sel[(sel['f0_mean_t'] > 0) & (sel['number_of_elements_processed'] > 0)].copy()

truth = (sel.sort_values('algorithm')
           .groupby(['sample_size', 'f0', 'seed', 'number_of_elements_processed'], as_index=False)
           .agg({'f0_mean_t': 'first'}))

truth['rho_final'] = (truth['sample_size'] / truth['f0']).astype(int)
truth['rho_t'] = truth['number_of_elements_processed'] / truth['f0_mean_t']
truth['progress_pct'] = 100.0 * truth['number_of_elements_processed'] / truth['sample_size']
truth['is_boundary'] = (truth['number_of_elements_processed'] % truth['rho_final'] == 0)
truth_boundary = truth[truth['is_boundary']].copy()

rho_values = sorted(truth['rho_final'].unique().tolist())

print('rows truth-deduplicated:', len(truth))
print('rows boundary-only:', len(truth_boundary))
print('rho final values:', rho_values)


In [ ]:
def _new_grid(n_panels: int):
    ncols = 3
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.2 * nrows), constrained_layout=True)
    axes = np.array(axes).reshape(-1)
    return fig, axes


def _plot_ratio_panels(data: pd.DataFrame,
                       title: str,
                       out_name: str,
                       boundary_mode: bool) -> Path:
    fig, axes = _new_grid(len(rho_values))

    for idx, rho in enumerate(rho_values):
        ax = axes[idx]
        sub = data[data['rho_final'] == rho].sort_values('number_of_elements_processed')
        if sub.empty:
            ax.axis('off')
            continue

        x = sub['progress_pct'].to_numpy(dtype=float)
        y = sub['rho_t'].to_numpy(dtype=float)

        if boundary_mode:
            ax.plot(x, y, color='#1f77b4', linewidth=1.7, marker='o', markersize=4.0,
                    label=r'$\rho_t$ (boundary-only)')
        else:
            ax.plot(x, y, color='#1f77b4', linewidth=2.0,
                    label=r'$\rho_t$ (checkpoint streaming)')

        ax.axhline(float(rho), color='#d62728', linestyle='--', linewidth=1.4,
                   label=fr'$\rho = n/d = {rho}$')

        ax.set_title(f'rho = {rho}')
        ax.set_xlabel('Progresso stream (%)')
        ax.set_ylabel(r'Rapporto $t/F_0(t)$')

    for j in range(len(rho_values), len(axes)):
        axes[j].axis('off')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.02), frameon=True)
    fig.suptitle(title, y=1.06)

    out = OUT_DIR / out_name
    fig.savefig(out, dpi=170, bbox_inches='tight')
    plt.close(fig)
    return out


out_full_linear = _plot_ratio_panels(
    data=truth,
    title='Rapporto campioni processati/distinti - curva completa (lineare)',
    out_name='ratio_processed_vs_distinct_streaming_linear.png',
    boundary_mode=False,
)
out_bound_linear = _plot_ratio_panels(
    data=truth_boundary,
    title='Rapporto campioni processati/distinti - boundary-only (lineare, t % rho == 0)',
    out_name='ratio_processed_vs_distinct_streaming_boundary_linear.png',
    boundary_mode=True,
)

print('saved:', out_full_linear)
print('saved:', out_bound_linear)
